In [ ]:
import torch
import torch.nn
import sklearn
import numpy as np
import matplotlib.pyplot as plt
from torch.optim import Adam
from torch.utils.data import Dataset,DataLoader
from torchsummary import summary
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd

In [ ]:
!pip install opendatasets
import opendatasets as od

od.download("https://www.kaggle.com/datasets/aneruddhadas/synthetic-data-interonit-hacks/settings")


Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
data_df = pd.read_csv("/content/synthetic-data-interonit-hacks/synthetic_resting_hr_dataset_3000.csv")
data_df = data_df[data_df['BPMLabel'] != 'Warning']
data_df = data_df.reset_index(drop=True)
data_df

In [ ]:
print(data_df['BPMLabel'])

In [ ]:
print(data_df["BPMLabel"].value_counts())

In [ ]:
X = np.array(data_df.iloc[:,:-1])
y = np.array(data_df.iloc[:,-1])

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [ ]:
X = np.array(data_df.iloc[:,:-1])
y = np.array(data_df.iloc[:,-1])

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
print("Mean:", scaler.mean_[0])
print("Std:", scaler.scale_[0])

In [ ]:
print("Mean:", scaler.mean_[0])
print("Std:", scaler.scale_[0])

In [ ]:
label_mapping_df = pd.DataFrame({
    'numerical_label': np.unique(y),
    'string_label': label_encoder.inverse_transform(np.unique(y))
})
display(label_mapping_df)

In [ ]:
y.dtype

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2)

In [ ]:
label_mapping_df = pd.DataFrame({
    'numerical_label': np.unique(y),
    'string_label': label_encoder.inverse_transform(np.unique(y))
})
display(label_mapping_df)

In [ ]:
class dataset(Dataset):
  def __init__(self,X,Y):
    self.X = torch.tensor(X,dtype = torch.float32).to(device)
    self.Y = torch.tensor(Y, dtype = torch.float32).unsqueeze(1).to(device)

  def __len__(self):
    return len(self.X)
  def __getitem__(self,index):
    return self.X[index],self.Y[index]

In [ ]:
training_data = dataset(X_train,y_train)

test_data = dataset(X_test,y_test)


In [ ]:
train_dataloader = DataLoader(dataset =training_data,
                              shuffle = True,
                              batch_size = 32,)

test_dataloader = DataLoader(dataset =test_data,
                              shuffle = False,
                              batch_size = 32,)

In [ ]:
from torch import nn

In [ ]:
class HeartRateModel(nn.Module):
  def __init__(self,
               in_features: int,
               hidden_units: int,
               out_features: int):
    super(). __init__()
    self.layers = nn.Sequential(
        nn.Linear(in_features = in_features, out_features= hidden_units),
        nn.Linear(in_features = hidden_units, out_features = out_features)
    )

  def forward(self,x):
    return self.layers(x)

In [ ]:
num_classes = 1
model = HeartRateModel(in_features = 1, hidden_units = 10, out_features = num_classes).to(device)
model

In [ ]:
optimizer = torch.optim.SGD(params = model.parameters(),
                             lr = 0.1)
loss_fn = nn.BCEWithLogitsLoss()

In [ ]:
from tqdm.auto import tqdm

In [ ]:
total_loss_train_plot = []

total_loss_test_plot = []

total_acc_train_plot = []

total_acc_test_plot = []

In [ ]:
epochs = 20

total_test_loss = 0.0

for epoch in tqdm(range(epochs)):
  total_acc_train = 0
  total_loss_train = 0
  total_acc_test = 0
  total_loss_test = 0

  for batch,(X,y) in enumerate(train_dataloader):

    model.train()
    y_logits = model(X)

    loss = loss_fn(y_logits, y)
    total_loss_train += loss.item()

    y_pred_probs = torch.sigmoid(y_logits)
    y_pred_labels = (y_pred_probs > 0.5).float()

    train_acc = accuracy_score(y_true = y.cpu(),
                         y_pred = y_pred_labels.cpu())
    total_acc_train += train_acc

    optimizer.zero_grad()

    loss.backward()
    optimizer.step()



  total_loss_train /= len(train_dataloader)
  total_loss_train_plot.append(total_loss_train)
  total_acc_train /= len(train_dataloader)
  total_acc_train_plot.append(total_acc_train)

  model.eval()
  with torch.inference_mode():
    for X_test, y_test in test_dataloader:
      test_logits = model(X_test)
      test_loss = loss_fn(test_logits, y_test)
      total_loss_test += test_loss.item()

      test_pred_probs = torch.sigmoid(test_logits)
      test_pred_labels = (test_pred_probs > 0.5).float()

      test_acc = accuracy_score(y_true = y_test.cpu(),
                                y_pred = test_pred_labels.cpu())
      total_acc_test += test_acc

  total_loss_test /= len(test_dataloader)
  total_loss_test_plot.append(total_loss_test)
  total_acc_test /= len(test_dataloader)
  total_acc_test_plot.append(total_acc_test)

  print(f"Epoch: {epoch+1} | Train_Loss: {total_loss_train:.4f} | Train_acc : {total_acc_train:.4f} | Test Loss: {total_loss_test:.4f} | Test Acc: {total_acc_test:.4f}")

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(total_loss_train_plot) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, total_loss_train_plot, label='Train Loss', linewidth=2)
plt.plot(epochs, total_loss_test_plot, label='Test Loss', linewidth=2)
plt.title('Training and Test Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(8, 5))
plt.plot(epochs, total_acc_train_plot, label='Train Accuracy', linewidth=2)
plt.plot(epochs, total_acc_test_plot, label='Test Accuracy', linewidth=2)
plt.title('Training and Test Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
!pip install --upgrade onnx onnxscript

In [ ]:
torch_model = HeartRateModel(in_features = 1, hidden_units = 10, out_features = num_classes).to(device)

example_inputs = (torch.randn(1,1).to(device),)
onnx_program = torch.onnx.export(torch_model, example_inputs, dynamo=True)

In [ ]:
onnx_program.save("heartrate.onnx")